# Attention Value Analysis

Analyze value vectors: norm profiles, weighted outputs, rank structure,
position variation, and unembedding projections.

## Why This Matters

Attention value reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_value_analysis import (
    value_vector_profile, value_weighted_output,
    value_rank_analysis, value_position_variation,
    value_unembed_projection,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Value Vector Profile

Norm, direction, and consistency of value vectors per head.

In [ ]:
for layer in range(4):
    result = value_vector_profile(model, tokens, layer=layer)
    print(f'Layer {layer}:')
    for h in result['per_head']:
        bar = '█' * int(h['direction_consistency'] * 20)
        print(f"  Head {h['head']}: norm={h['mean_norm']:.4f}±{h['std_norm']:.4f}, "
              f"consistency={h['direction_consistency']:.4f} {bar}")
    print()

## Value Weighted Output

Decompose the attention-weighted output by source contribution.

In [ ]:
result = value_weighted_output(model, tokens, layer=0, head=0, position=-1)
print(f"Weighted output norm: {result['weighted_output_norm']:.4f}\n")
print('Top source contributions:')
for s in result['per_source'][:5]:
    bar = '█' * int(s['fraction'] * 30)
    print(f"  Src {s['source']}: attn={s['attention_weight']:.4f}, "
          f"v_norm={s['value_norm']:.4f}, "
          f"contrib={s['contribution_norm']:.4f} ({s['fraction']:.1%}) {bar}")

## Value Rank Analysis

Effective rank of value vectors — low rank means constrained subspace.

In [ ]:
for layer in range(4):
    result = value_rank_analysis(model, tokens, layer=layer)
    print(f"Layer {layer} (mean rank={result['mean_rank']:.2f}):")
    for h in result['per_head']:
        low = ' [LOW RANK]' if h['is_low_rank'] else ''
        print(f"  Head {h['head']}: eff_rank={h['effective_rank']:.2f}, "
              f"top_sv={h['top_sv_fraction']:.1%}{low}")
    print()

## Position Variation

Do value vectors change across positions, or stay constant?

In [ ]:
for layer in range(4):
    result = value_position_variation(model, tokens, layer=layer)
    print(f"Layer {layer} ({result['n_position_dependent']} position-dependent):")
    for h in result['per_head']:
        dep = ' [POS-DEP]' if h['is_position_dependent'] else ''
        print(f"  Head {h['head']}: dir_var={h['direction_variation']:.4f}, "
              f"norm_cv={h['norm_cv']:.4f}{dep}")
    print()

## Value-Unembed Projection

What tokens would value vectors promote through OV and unembedding?

In [ ]:
result = value_unembed_projection(model, tokens, layer=0, head=0, position=-1)
for s in result['per_source']:
    top = ', '.join(f"tok{t['token']}({t['logit']:+.3f})" for t in s['top_tokens'])
    bot = ', '.join(f"tok{t['token']}({t['logit']:+.3f})" for t in s['bottom_tokens'])
    print(f"  Src {s['source']} (tok {s['source_token']}): norm={s['output_norm']:.4f}")
    print(f"    Promotes: {top}")
    print(f"    Suppresses: {bot}")